## 4단계 검수: 전체 테스트와 실제 이미지 batch

이 Notebook은 다음 두 가지를 직접 확인합니다.

1. 프로젝트의 전체 단위 테스트를 실행해 모든 항목이 통과하는지 확인
2. 실제 Train/Validation batch의 이미지 4장, tensor shape, Answer와 class target을 눈으로 확인

Test 데이터는 읽지 않습니다.

In [ ]:
import io
import sys
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'train.csv').exists():
    ROOT = ROOT.parent

assert (ROOT / 'data' / 'train.csv').exists(), '프로젝트 루트를 찾지 못했습니다.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'프로젝트 루트: {ROOT}')
print('범위: TRAIN ONLY')

## 1. 실제 전체 테스트 실행

`tests/test_*.py`를 자동 탐색하여 터미널에서 실행하는 것과 같은 테스트를 실행합니다. 테스트가 하나라도 실패하면 마지막 `assert`에서 중단됩니다.

In [ ]:
test_loader = unittest.TestLoader()
test_suite = test_loader.discover(str(ROOT / 'tests'), pattern='test_*.py')
discovered_test_count = test_suite.countTestCases()
test_output = io.StringIO()
test_result = unittest.TextTestRunner(stream=test_output, verbosity=2).run(test_suite)

print(test_output.getvalue())
print(f'발견한 테스트: {discovered_test_count}개')
print(f'실행한 테스트: {test_result.testsRun}개')
print(f'실패: {len(test_result.failures)}개')
print(f'오류: {len(test_result.errors)}개')
print(f'건너뜀: {len(test_result.skipped)}개')

assert test_result.wasSuccessful(), '통과하지 못한 테스트가 있습니다.'
assert test_result.testsRun == discovered_test_count
print('✓ 발견된 모든 테스트가 통과했습니다.')

## 2. 실제 Train/Validation batch 생성

고정된 분할 ID와 4단계 Dataset을 사용합니다. Train은 seed 42로 shuffle하고, Validation은 원래 분할 파일 순서를 유지합니다.

In [ ]:
from src.dataset import (
    ChallengeDataset,
    ImagePreprocessConfig,
    build_dataloader,
    build_image_transform,
    decode_class,
)

transform = build_image_transform(
    ImagePreprocessConfig(image_size=224, resize_size=256)
)
train_dataset = ChallengeDataset.from_split_csv(
    data_dir=ROOT / 'data',
    split_csv=ROOT / 'outputs' / 'splits' / 'train_ids.csv',
    expected_split='train',
    transform=transform,
)
validation_dataset = ChallengeDataset.from_split_csv(
    data_dir=ROOT / 'data',
    split_csv=ROOT / 'outputs' / 'splits' / 'validation_ids.csv',
    expected_split='validation',
    transform=transform,
)
train_loader = build_dataloader(
    train_dataset, batch_size=4, shuffle=True, seed=42, num_workers=0
)
validation_loader = build_dataloader(
    validation_dataset, batch_size=4, shuffle=False, seed=42, num_workers=0
)
train_batch = next(iter(train_loader))
validation_batch = next(iter(validation_loader))

print(f'Train samples: {len(train_dataset):,}')
print(f'Validation samples: {len(validation_dataset):,}')
print(f'Train images shape: {tuple(train_batch["images"].shape)}')
print(f'Validation images shape: {tuple(validation_batch["images"].shape)}')

## 3. batch 자동 검사

이미지 shape, Answer shape, class target 복원, NaN/무한값을 검사합니다.

In [ ]:
def verify_batch(batch, name):
    batch_size = len(batch['sample_ids'])
    assert tuple(batch['images'].shape) == (batch_size, 4, 3, 224, 224)
    assert tuple(batch['answers'].shape) == (batch_size, 4)
    assert tuple(batch['targets'].shape) == (batch_size,)
    assert torch.isfinite(batch['images']).all()

    for answer, target in zip(batch['answers'], batch['targets'], strict=True):
        assert decode_class(int(target.item())) == tuple(answer.tolist())

    print(f'✓ {name}: shape, 유한한 pixel, label 왕복 검사 통과')

verify_batch(train_batch, 'Train batch')
verify_batch(validation_batch, 'Validation batch')

## 4. 실제 이미지 4장 확인

전처리된 tensor를 사람이 볼 수 있도록 정규화만 되돌려 표시합니다. 이미지 순서는 모델에 들어가는 `Input_1~Input_4` 그대로입니다. 제목의 `time position`은 각 Input의 정답 시간 위치입니다.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def show_batch_sample(batch, batch_index, title):
    sample_id = batch['sample_ids'][batch_index]
    sentence = batch['sentences'][batch_index]
    input_names = batch['input_names'][batch_index]
    images = batch['images'][batch_index]
    answer = batch['answers'][batch_index].tolist()
    target = int(batch['targets'][batch_index].item())
    decoded = list(decode_class(target))
    chronological_inputs = [
        input_index
        for input_index, _ in sorted(
            enumerate(answer, start=1), key=lambda item: item[1]
        )
    ]

    print('\n' + '=' * 100)
    print(title)
    print(f'Id: {sample_id}')
    print(f'Caption: {sentence}')
    print(f'images tensor: {tuple(images.shape)}')
    print(f'Answer: {answer}')
    print(f'class target: {target}')
    print(f'decoded target: {decoded}')
    print(f'시간순 Input 번호: {chronological_inputs}')
    print(f'label 왕복 일치: {answer == decoded}')

    fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
    for input_index, axis in enumerate(axes, start=1):
        display_image = (
            images[input_index - 1] * IMAGENET_STD + IMAGENET_MEAN
        ).clamp(0, 1)
        axis.imshow(display_image.permute(1, 2, 0).cpu())
        axis.set_title(
            f'Input_{input_index} | time position={answer[input_index - 1]}\n'
            f'{input_names[input_index - 1]}',
            fontsize=9,
        )
        axis.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
show_batch_sample(train_batch, 0, 'Train batch의 첫 샘플')
show_batch_sample(validation_batch, 0, 'Validation batch의 첫 샘플')

## 확인할 항목

- 전체 테스트 목록이 모두 `ok`인지
- 마지막 결과가 `Ran 20 tests`, `OK`인지
- batch shape가 `(4, 4, 3, 224, 224)`인지
- 이미지가 `Input_1~Input_4` 순서로 표시되는지
- `Answer`와 `decoded target`이 같은지
- `label 왕복 일치`가 `True`인지